# Configurable NLP Workbench (RTX 3050-ready)

A practical, configurable workflow for **text in tabular datasets**. Start by editing the configuration cell, then run top-to-bottom.

**Supported workflows**
- Single-label text classification
- Multilabel text classification
- Text regression
- Unsupervised text clustering (no target column)
- Optional transformer sentence embeddings using a CUDA-capable GPU

**Important scope:** this is a strong general-purpose workbench for *text represented in rows/columns*. It is not a universal solution for generative AI, translation, NER, question answering, or document-layout extraction; those need task-specific architectures and evaluation.

The notebook begins with explainable EDA recommendations, establishes a TF-IDF baseline, optionally compares transformer embeddings, evaluates appropriately, and saves the fitted pipeline. GPU acceleration is used where supported; classical scikit-learn models generally run on CPU and can still be faster/better for small datasets.

## 0. Install dependencies
Run once in the environment/kernel you intend to use. Restart the kernel if prompted.

In [ ]:
%pip install -U pandas numpy scipy scikit-learn matplotlib seaborn joblib ipywidgets
%pip install -U torch sentence-transformers

## 1. Configuration — edit this cell first
Set `DATA_PATH`, `TEXT_COLUMNS`, and (for supervised tasks) `TARGET_COLUMN`. Use `TASK='auto'` for a first pass, but explicitly set the task when you know it. For clustering, set `TARGET_COLUMN = None`.

In [ ]:
from pathlib import Path

# ---------- DATA ----------
DATA_PATH = "your_dataset.csv"       # CSV or Parquet
CSV_SEPARATOR = ","
ENCODING = "utf-8"
TEXT_COLUMNS = ["text"]               # one or more text columns; e.g. ["title", "body"]
TARGET_COLUMN = "label"               # None for unsupervised clustering

# ---------- TASK ----------
# "auto", "classification", "multilabel", "regression", "clustering"
TASK = "auto"

# ---------- DATA SPLITTING / REPRODUCIBILITY ----------
TEST_SIZE = 0.20
RANDOM_STATE = 42
CV_FOLDS = 5                          # automatically reduced for small classes
MAX_TFIDF_FEATURES = 100_000
NGRAM_RANGE = (1, 2)
MIN_DF = 2                            # set to 1 for very small datasets

# ---------- OPTIONAL TRANSFORMER EMBEDDINGS ----------
USE_TRANSFORMERS = True
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 32              # lower to 8 or 16 if CUDA runs out of memory
NORMALIZE_EMBEDDINGS = True

# ---------- COMPUTE ----------
PREFER_GPU = True
# Optional: limit rows for quick iteration; None means use all rows
MAX_ROWS = None

# ---------- OUTPUT ----------
OUTPUT_DIR = Path("nlp_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_PATH = OUTPUT_DIR / "best_nlp_model.joblib"
PREDICTIONS_PATH = OUTPUT_DIR / "test_predictions.csv"

print("Configuration loaded.")

## 2. Imports and hardware check

In [ ]:
import os, re, json, warnings, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, classification_report,
    confusion_matrix, mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_fscore_support, silhouette_score
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import chi2
import joblib

warnings.filterwarnings("ignore", category=UserWarning)
RNG = np.random.RandomState(RANDOM_STATE)

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    print("PyTorch:", torch.__version__)
    print("CUDA available:", CUDA_AVAILABLE)
    if CUDA_AVAILABLE:
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
except Exception as e:
    CUDA_AVAILABLE = False
    print("PyTorch/CUDA unavailable:", repr(e))

DEVICE = "cuda" if (PREFER_GPU and CUDA_AVAILABLE) else "cpu"
print("Transformer device:", DEVICE)
print("Python:", platform.python_version())

## 3. Load data and validate configuration

In [ ]:
path = Path(DATA_PATH)
if not path.exists():
    raise FileNotFoundError(f"Cannot find {path.resolve()}. Update DATA_PATH in Section 1.")

if path.suffix.lower() in [".parquet", ".pq"]:
    df = pd.read_parquet(path)
else:
    df = pd.read_csv(path, sep=CSV_SEPARATOR, encoding=ENCODING)

if MAX_ROWS is not None:
    df = df.sample(min(MAX_ROWS, len(df)), random_state=RANDOM_STATE).reset_index(drop=True)

if not TEXT_COLUMNS:
    raise ValueError("Set at least one column in TEXT_COLUMNS.")
missing = [c for c in TEXT_COLUMNS if c not in df.columns]
if missing:
    raise KeyError(f"Text columns not found: {missing}")
if TARGET_COLUMN is not None and TARGET_COLUMN not in df.columns:
    raise KeyError(f"Target column {TARGET_COLUMN!r} not found.")

print("Shape:", df.shape)
display(df.head())
print("\\nColumn types:")
display(df.dtypes.to_frame("dtype"))

## 4. EDA and decision support
This section does not silently modify your data. It reports issues and gives recommendations for you to accept/reject.

In [ ]:
def combine_text(frame, columns):
    # Preserve missing values as empty strings and make a single model input.
    return frame[columns].fillna("").astype(str).agg(" [SEP] ".join, axis=1).str.strip()

eda = []
for col in df.columns:
    s = df[col]
    missing_pct = s.isna().mean() * 100
    unique = s.nunique(dropna=True)
    unique_pct = unique / max(s.notna().sum(), 1) * 100
    row = {
        "column": col, "dtype": str(s.dtype), "missing_%": round(missing_pct, 2),
        "unique": int(unique), "unique_%_nonmissing": round(unique_pct, 2)
    }
    if col in TEXT_COLUMNS:
        lengths = s.fillna("").astype(str).str.split().str.len()
        row.update({"mean_words": round(lengths.mean(), 1), "median_words": round(lengths.median(), 1),
                    "empty_%": round((lengths == 0).mean()*100, 2)})
    eda.append(row)
eda_df = pd.DataFrame(eda)
display(eda_df)

print("\\nAutomated recommendations:")
for _, r in eda_df.iterrows():
    advice = []
    if r["missing_%"] > 30: advice.append("high missingness: investigate whether missingness is meaningful")
    elif r["missing_%"] > 0: advice.append("some missing values: handle explicitly")
    if r["unique_%_nonmissing"] > 98 and r["unique"] > 100:
        advice.append("near-unique: likely ID/free text; do not use as a categorical feature without a reason")
    if r["column"] in TEXT_COLUMNS:
        advice.append("text feature: inspect examples, length distribution, language, duplicates, and leakage")
    if r["column"] == TARGET_COLUMN:
        advice.append("target: inspect class balance/label quality and ensure it is not present in input text")
    if not advice: advice.append("no immediate automated warning")
    print(f"- {r['column']}: " + "; ".join(advice))

for c in TEXT_COLUMNS:
    print(f"\\nExamples from {c}:")
    display(df[c].dropna().astype(str).sample(min(5, df[c].notna().sum()), random_state=RANDOM_STATE).to_frame())

text_all = combine_text(df, TEXT_COLUMNS)
word_lengths = text_all.str.split().str.len()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(word_lengths.clip(upper=word_lengths.quantile(.99)), bins=40, ax=ax[0])
ax[0].set_title("Text length (words; clipped at 99th percentile)")
sns.histplot(text_all.str.len().clip(upper=text_all.str.len().quantile(.99)), bins=40, ax=ax[1])
ax[1].set_title("Text length (characters; clipped at 99th percentile)")
plt.tight_layout(); plt.show()

print("Exact duplicate text rows:", int(text_all.duplicated().sum()))
print("Potential empty text rows:", int((text_all.str.strip() == "").sum()))

### Target diagnostics (supervised tasks)

In [ ]:
if TARGET_COLUMN is not None:
    target = df[TARGET_COLUMN]
    print("Target missing:", target.isna().sum(), f"({target.isna().mean()*100:.2f}%)")
    display(target.value_counts(dropna=False).head(30).to_frame("count"))
    if target.nunique(dropna=True) <= 30:
        target.value_counts(dropna=False).plot(kind="bar", figsize=(10, 3), title="Target distribution")
        plt.tight_layout(); plt.show()
    print("Decision guide:")
    print("- One label per row -> classification.")
    print("- Multiple labels per row (list/string convention) -> multilabel; parse labels explicitly below.")
    print("- Continuous numeric outcome -> regression.")
    print("- No target -> clustering / exploratory grouping.")
else:
    print("No target configured: unsupervised workflow will be used.")

## 5. Choose task and prepare labels
`auto` is only a heuristic. Set `TASK` manually if it chooses incorrectly. For multilabel, `TARGET_COLUMN` must contain Python-like lists or strings separated by `|` (edit `MULTILABEL_SEPARATOR` if needed).

In [ ]:
MULTILABEL_SEPARATOR = "|"

def infer_task(y):
    if y is None: return "clustering"
    if pd.api.types.is_numeric_dtype(y):
        nunique = y.nunique(dropna=True)
        # Numeric targets with few distinct values are often class IDs.
        return "classification" if nunique <= 20 else "regression"
    return "classification"

task = infer_task(df[TARGET_COLUMN] if TARGET_COLUMN else None) if TASK == "auto" else TASK
if task not in {"classification", "multilabel", "regression", "clustering"}:
    raise ValueError("Unsupported TASK.")
if task == "clustering" and TARGET_COLUMN is not None:
    print("Clustering selected: TARGET_COLUMN will be ignored.")
if task != "clustering" and TARGET_COLUMN is None:
    raise ValueError("A target column is required for supervised tasks.")

X_text = combine_text(df, TEXT_COLUMNS)
valid = X_text.str.strip().ne("")
if task != "clustering":
    valid &= df[TARGET_COLUMN].notna()
X_text = X_text[valid].reset_index(drop=True)
source_rows = df.loc[valid].reset_index(drop=True)

if task == "multilabel":
    from sklearn.preprocessing import MultiLabelBinarizer
    def parse_labels(v):
        if isinstance(v, (list, tuple, set)): return list(v)
        return [x.strip() for x in str(v).split(MULTILABEL_SEPARATOR) if x.strip()]
    label_lists = source_rows[TARGET_COLUMN].apply(parse_labels)
    mlb = MultiLabelBinarizer()
    y = mlb.fit_transform(label_lists)
    print("Labels:", list(mlb.classes_))
elif task == "classification":
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(source_rows[TARGET_COLUMN].astype(str))
    print("Classes:", list(label_encoder.classes_))
elif task == "regression":
    y = pd.to_numeric(source_rows[TARGET_COLUMN], errors="coerce").to_numpy()
    keep = np.isfinite(y)
    X_text = X_text[keep].reset_index(drop=True)
    source_rows = source_rows.loc[keep].reset_index(drop=True)
    y = y[keep]
else:
    y = None

print("Task:", task, "| usable rows:", len(X_text), "| text columns:", TEXT_COLUMNS)
if len(X_text) < 10:
    print("WARNING: Very small dataset. Metrics and train/test split may be unstable.")

## 6. Split data
For classification, stratification is used where possible. For multilabel, a simple random split is used; use iterative stratification for highly imbalanced multilabel datasets.

In [ ]:
indices = np.arange(len(X_text))
if task == "classification":
    counts = pd.Series(y).value_counts()
    stratify = y if counts.min() >= 2 else None
else:
    stratify = None

if task == "clustering":
    X_train_text, X_test_text = X_text, None
    y_train, y_test = None, None
else:
    X_train_text, X_test_text, y_train, y_test = train_test_split(
        X_text, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify
    )

print("Train rows:", len(X_train_text))
if X_test_text is not None: print("Test rows:", len(X_test_text))

## 7. TF-IDF baseline
A strong, fast, interpretable baseline for many classification/regression problems. It uses word n-grams; do not remove stopwords or stem automatically—those choices can hurt meaning, especially negation and domain terminology.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=MAX_TFIDF_FEATURES,
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    sublinear_tf=True,
    strip_accents="unicode",
    dtype=np.float32
)
Xtr = tfidf.fit_transform(X_train_text)
Xte = tfidf.transform(X_test_text) if X_test_text is not None else None
print("TF-IDF matrix:", Xtr.shape, "| sparsity:", round(1 - Xtr.nnz/(Xtr.shape[0]*Xtr.shape[1]), 4))

## 8. Train and compare classical models

In [ ]:
from sklearn.base import clone

if task == "classification":
    candidates = {
        "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "LinearSVC": LinearSVC(class_weight="balanced"),
        "ComplementNB": ComplementNB()
    }
elif task == "multilabel":
    candidates = {
        "OneVsRest LogisticRegression": OneVsRestClassifier(LogisticRegression(max_iter=2000)),
        "OneVsRest LinearSVC": OneVsRestClassifier(LinearSVC())
    }
elif task == "regression":
    candidates = {
        "Ridge": Ridge(alpha=1.0),
        "Ridge alpha=10": Ridge(alpha=10.0)
    }
else:
    candidates = {}

results = []
fitted_models = {}
if task != "clustering":
    for name, model in candidates.items():
        try:
            model.fit(Xtr, y_train)
            pred = model.predict(Xte)
            if task == "classification":
                score = f1_score(y_test, pred, average="macro", zero_division=0)
                metric_name = "macro_f1"
            elif task == "multilabel":
                score = f1_score(y_test, pred, average="micro", zero_division=0)
                metric_name = "micro_f1"
            else:
                score = mean_absolute_error(y_test, pred)
                metric_name = "MAE (lower is better)"
            results.append({"model": name, metric_name: score})
            fitted_models[name] = model
            print(name, metric_name, round(score, 5))
        except Exception as e:
            print("Skipped", name, "because:", repr(e))

    if not results:
        raise RuntimeError("No candidate model trained successfully.")
    results_df = pd.DataFrame(results)
    display(results_df)
    best_name = (results_df.iloc[results_df[metric_name].argmin()]["model"]
                 if task == "regression" else results_df.iloc[results_df[metric_name].argmax()]["model"])
    best_model = fitted_models[best_name]
    print("Best baseline:", best_name)

## 9. Evaluate the baseline
Always inspect per-class results and errors, not just one aggregate score. For imbalanced classes, macro-F1 and balanced accuracy are often more informative than accuracy.

In [ ]:
if task == "classification":
    pred = best_model.predict(Xte)
    print(classification_report(y_test, pred, target_names=label_encoder.classes_, zero_division=0))
    cm = confusion_matrix(y_test, pred)
    plt.figure(figsize=(7, 6)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()
elif task == "multilabel":
    pred = best_model.predict(Xte)
    print("Micro F1:", f1_score(y_test, pred, average="micro", zero_division=0))
    print("Macro F1:", f1_score(y_test, pred, average="macro", zero_division=0))
    print("Hamming loss:", __import__("sklearn.metrics").metrics.hamming_loss(y_test, pred))
elif task == "regression":
    pred = best_model.predict(Xte)
    print("MAE:", mean_absolute_error(y_test, pred))
    print("RMSE:", mean_squared_error(y_test, pred)**0.5)
    print("R2:", r2_score(y_test, pred))
    plt.figure(figsize=(5, 5)); plt.scatter(y_test, pred, alpha=.5)
    plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title("Actual vs predicted")
    plt.show()

## 10. Optional transformer sentence embeddings (uses RTX 3050 when CUDA is available)
This embeds text using a pretrained sentence-transformer, then trains a lightweight downstream model. The transformer weights are downloaded on first use. Reduce `EMBEDDING_BATCH_SIZE` if you run out of VRAM. Embeddings are not fine-tuned here; this is a feature-extraction comparison, not full task-specific transformer fine-tuning.

In [ ]:
transformer_available = False
transformer_results = []
if USE_TRANSFORMERS:
    try:
        from sentence_transformers import SentenceTransformer
        encoder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
        E_train = encoder.encode(
            X_train_text.tolist(), batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=True, convert_to_numpy=True,
            normalize_embeddings=NORMALIZE_EMBEDDINGS
        ).astype(np.float32)
        E_test = encoder.encode(
            X_test_text.tolist(), batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=True, convert_to_numpy=True,
            normalize_embeddings=NORMALIZE_EMBEDDINGS
        ).astype(np.float32) if X_test_text is not None else None
        transformer_available = True
        print("Embedding shapes:", E_train.shape, None if E_test is None else E_test.shape)
    except Exception as e:
        print("Transformer stage unavailable; continuing with TF-IDF baseline.")
        print(type(e).__name__, str(e)[:500])

In [ ]:
if transformer_available and task != "clustering":
    if task == "classification":
        emb_model = LogisticRegression(max_iter=2000, class_weight="balanced")
        emb_model.fit(E_train, y_train)
        emb_pred = emb_model.predict(E_test)
        emb_score = f1_score(y_test, emb_pred, average="macro", zero_division=0)
        emb_metric = "macro_f1"
    elif task == "multilabel":
        emb_model = OneVsRestClassifier(LogisticRegression(max_iter=2000))
        emb_model.fit(E_train, y_train)
        emb_pred = emb_model.predict(E_test)
        emb_score = f1_score(y_test, emb_pred, average="micro", zero_division=0)
        emb_metric = "micro_f1"
    else:
        emb_model = Ridge(alpha=1.0)
        emb_model.fit(E_train, y_train)
        emb_pred = emb_model.predict(E_test)
        emb_score = mean_absolute_error(y_test, emb_pred)
        emb_metric = "MAE (lower is better)"
    print("Transformer embeddings:", emb_metric, round(emb_score, 5))
    print("Compare with TF-IDF:", metric_name, round(float(results_df.iloc[0][metric_name]), 5))

## 11. Unsupervised clustering (only when TASK='clustering')
The number of clusters is a modeling choice, not something EDA can reliably infer. Compare silhouette scores across a sensible range and inspect representative texts.

In [ ]:
if task == "clustering":
    # Use transformer embeddings when available; otherwise TF-IDF + SVD.
    if transformer_available:
        cluster_X = E_train
        cluster_method = "sentence-transformer embeddings"
    else:
        reducer = TruncatedSVD(n_components=min(100, Xtr.shape[1]-1), random_state=RANDOM_STATE)
        cluster_X = reducer.fit_transform(Xtr)
        cluster_method = "TF-IDF + SVD"
    cluster_scores = []
    models = {}
    for k in range(2, min(11, len(cluster_X))):
        km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10, batch_size=1024)
        labels = km.fit_predict(cluster_X)
        sample_n = min(5000, len(cluster_X))
        score = silhouette_score(cluster_X, labels, sample_size=sample_n, random_state=RANDOM_STATE)
        cluster_scores.append((k, score)); models[k] = km
    display(pd.DataFrame(cluster_scores, columns=["k", "silhouette"]))
    best_k = max(cluster_scores, key=lambda x: x[1])[0]
    cluster_model = models[best_k]
    cluster_labels = cluster_model.labels_
    print("Selected k:", best_k, "| method:", cluster_method)
    clustered = pd.DataFrame({"text": X_text, "cluster": cluster_labels})
    for c in sorted(clustered.cluster.unique()):
        print(f"\\nCluster {c} examples:")
        display(clustered[clustered.cluster == c].head(5))

## 12. Error analysis and feature inspection

In [ ]:
if task == "classification":
    error_df = pd.DataFrame({
        "text": X_test_text.reset_index(drop=True),
        "actual": label_encoder.inverse_transform(y_test),
        "predicted": label_encoder.inverse_transform(best_model.predict(Xte))
    })
    display(error_df[error_df.actual != error_df.predicted].head(30))
    # Top positive/negative weighted terms for linear models
    if hasattr(best_model, "coef_"):
        feature_names = np.array(tfidf.get_feature_names_out())
        coefs = best_model.coef_
        if coefs.shape[0] == 1 and len(label_encoder.classes_) == 2:
            for label, weights in [(label_encoder.classes_[1], coefs[0]), (label_encoder.classes_[0], -coefs[0])]:
                top = np.argsort(weights)[-20:][::-1]
                print(f"Terms associated with {label}:")
                display(pd.DataFrame({"term": feature_names[top], "weight": weights[top]}))
        else:
            for i, label in enumerate(label_encoder.classes_[:10]):
                top = np.argsort(coefs[i])[-15:][::-1]
                print(f"Terms associated with {label}:")
                display(pd.DataFrame({"term": feature_names[top], "weight": coefs[i][top]}))
elif task == "regression":
    error_df = pd.DataFrame({"text": X_test_text.reset_index(drop=True), "actual": y_test,
                             "predicted": best_model.predict(Xte)})
    error_df["absolute_error"] = (error_df.actual - error_df.predicted).abs()
    display(error_df.sort_values("absolute_error", ascending=False).head(30))

## 13. Choose final model and save artifacts
The selection below compares held-out test scores. For publication-quality estimates, keep a final untouched test set and select/tune models using cross-validation on training data only. Do not repeatedly optimize against the test set.

In [ ]:
if task != "clustering":
    chosen_family = "tfidf"
    final_model = best_model
    final_vectorizer = tfidf
    baseline_score = (results_df[metric_name].min() if task == "regression" else results_df[metric_name].max())
    if transformer_available:
        # Choose embeddings only if they improve the same metric.
        if task == "classification":
            use_embedding = emb_score > float(baseline_score)
        elif task == "multilabel":
            use_embedding = emb_score > float(baseline_score)
        else:
            use_embedding = emb_score < float(baseline_score)
        if use_embedding:
            chosen_family = "transformer_embeddings"
            final_model = emb_model
            final_vectorizer = None

    artifact = {
        "task": task, "text_columns": TEXT_COLUMNS, "target_column": TARGET_COLUMN,
        "family": chosen_family, "model": final_model, "vectorizer": final_vectorizer,
        "label_encoder": globals().get("label_encoder"),
        "multilabel_binarizer": globals().get("mlb"),
        "embedding_model": EMBEDDING_MODEL if chosen_family == "transformer_embeddings" else None,
        "normalize_embeddings": NORMALIZE_EMBEDDINGS
    }
    joblib.dump(artifact, MODEL_PATH)
    print("Saved:", MODEL_PATH.resolve(), "| family:", chosen_family)

    if task == "classification":
        final_pred = final_model.predict(E_test if chosen_family == "transformer_embeddings" else Xte)
        decoded = label_encoder.inverse_transform(final_pred)
        output = pd.DataFrame({"actual": label_encoder.inverse_transform(y_test), "predicted": decoded})
    elif task == "multilabel":
        final_pred = final_model.predict(E_test if chosen_family == "transformer_embeddings" else Xte)
        output = pd.DataFrame({
            "actual": [list(mlb.classes_[np.where(row)[0]]) for row in y_test],
            "predicted": [list(mlb.classes_[np.where(row)[0]]) for row in final_pred]
        })
    else:
        final_pred = final_model.predict(E_test if chosen_family == "transformer_embeddings" else Xte)
        output = pd.DataFrame({"actual": y_test, "predicted": final_pred})
    output.to_csv(PREDICTIONS_PATH, index=False)
    print("Predictions saved:", PREDICTIONS_PATH.resolve())
else:
    joblib.dump({"task": "clustering", "model": cluster_model,
                 "text_columns": TEXT_COLUMNS, "family": cluster_method,
                 "embedding_model": EMBEDDING_MODEL if transformer_available else None,
                 "vectorizer": None if transformer_available else tfidf,
                 "reducer": None if transformer_available else reducer}, MODEL_PATH)
    print("Saved clustering artifact:", MODEL_PATH.resolve())

## 14. Predict on new text
Run after training in this kernel. For transformer artifacts, the same embedding model is loaded and used. Input must provide the same text columns.

In [ ]:
def predict_new(data, artifact_path=MODEL_PATH):
    art = joblib.load(artifact_path)
    new_df = pd.DataFrame(data)
    missing_cols = [c for c in art["text_columns"] if c not in new_df.columns]
    if missing_cols: raise KeyError(f"Missing required text columns: {missing_cols}")
    new_text = combine_text(new_df, art["text_columns"])
    if art["task"] == "clustering":
        if art["family"] == "sentence-transformer embeddings":
            from sentence_transformers import SentenceTransformer
            enc = SentenceTransformer(art["embedding_model"], device=DEVICE)
            feats = enc.encode(new_text.tolist(), convert_to_numpy=True,
                               normalize_embeddings=True).astype(np.float32)
        else:
            feats = art["reducer"].transform(art["vectorizer"].transform(new_text))
        return art["model"].predict(feats)

    if art["family"] == "tfidf":
        feats = art["vectorizer"].transform(new_text)
    else:
        from sentence_transformers import SentenceTransformer
        enc = SentenceTransformer(art["embedding_model"], device=DEVICE)
        feats = enc.encode(new_text.tolist(), batch_size=EMBEDDING_BATCH_SIZE,
                           convert_to_numpy=True,
                           normalize_embeddings=art["normalize_embeddings"]).astype(np.float32)
    pred = art["model"].predict(feats)
    if art["task"] == "classification":
        return art["label_encoder"].inverse_transform(pred)
    if art["task"] == "multilabel":
        mlb_obj = art["multilabel_binarizer"]
        return [list(mlb_obj.classes_[np.where(row)[0]]) for row in pred]
    return pred

# Example:
# predict_new([{"text": "A sample sentence to classify."}])

## 15. Practical decision checklist

Before trusting results:
1. **Target validity:** are labels consistent, meaningful, and available at prediction time?
2. **Leakage:** remove IDs, post-outcome fields, duplicated target text, and features that reveal the answer.
3. **Duplicates:** check exact and near-duplicates across train/test; group-split when rows from the same author/customer/document family recur.
4. **Split strategy:** use time-based splits for future prediction; group splits for repeated entities; stratification for imbalanced classification.
5. **Metric:** choose based on the real cost of errors (macro-F1, PR-AUC, recall@precision, MAE, etc.).
6. **Language/domain:** verify tokenizer and embedding model fit your language and specialized vocabulary. The default MiniLM model is a general English-oriented baseline.
7. **GPU reality:** RTX 3050 VRAM varies by laptop/desktop configuration. Batch size is the first control to reduce. TF-IDF and these linear models mostly use CPU.
8. **Robustness:** evaluate on data from a later period or a different source, not just a random split.

**Recommended next step:** after this baseline works, add task-specific fine-tuning (e.g. Hugging Face Transformers) only if the baseline and error analysis show a real need. Keep a simple baseline as a reference.